# E2Q Proxy Full Benchmark

## tl;dr

**No result or verdict is pre-filled. Run this notebook top-to-bottom.**

This is the confirmatory multi-dataset benchmark for the already-frozen calibration-weighted two-qubit exposure proxy

\[
E_{2q,t}(a)=\sum_{g\in 2Q(a)}-\log\bigl(1-\epsilon_{g,t}\bigr).
\]

The benchmark does **not** redesign or tune E2Q. It tests whether the previously validated relationship between E2Q and actual noise-induced probability distortion generalizes across:

- **3 datasets:** Breast Cancer, Iris binary, Digits 0-vs-1;
- **3 IBM backends:** `ibm_fez`, `ibm_kingston`, `ibm_marrakesh`;
- **4 frozen historical calibration states/backend** spanning the prior observation window;
- the exact **16-architecture panel** frozen before the new Iris/Digits experiments;
- **5 independent ideal-training initializations**, with 3 prespecified seeds used for exact noisy validation.

The notebook reuses compact historical calibration snapshots already collected in the earlier confirmed experiments, so it requires **no IBM login** and never builds a full-device Aer noise model.

## Context & Methods

### Confirmatory hypotheses — frozen before this benchmark

**H1 — predictive validity.** E2Q should be positively associated with exact noisy probability distortion across most of the 9 dataset × backend cells.

**H2 — calibration adds value.** E2Q should outperform static structural complexity proxies, especially two-qubit gate count \(N_{2q}\) and logical depth.

**H3 — temporal sensitivity.** Within a fixed architecture, calibration-to-calibration changes in E2Q should track calibration-to-calibration changes in noisy probability distortion.

Primary noisy outcome:

\[
D(a,t)=\frac{1}{N}\sum_i\left|p_i^{ideal}-p_{i,t}^{noisy}\right|.
\]

Secondary outcomes: probability RMSE, log-loss degradation, Brier degradation, AUC degradation, and balanced-accuracy degradation.

### Frozen benchmark design

- Fixed split seed: **17** for every dataset. Data splits and preprocessing do not vary across training seeds.
- Ideal-training seeds: **[17, 29, 43, 71, 101]**.
- Exact noisy-validation seeds: **[17, 43, 101]**. Because Aer is run with exact density-matrix probabilities, these seeds represent independent trained parameterizations rather than shot noise.
- Training protocol: the same lightweight random-coordinate optimizer used in the successful pilot, **20 iterations**, with train/validation/test caps **160/80/80**.
- Noisy test subset: at most **64 stratified test cases/dataset**, frozen by the data split seed.
- Historical calibration indices: **[0, 2, 4, 7]** from the already-frozen 8-state manifests.
- Backends and physical path remain frozen at **[0,1,2,3]**. No path or backend is reselected.
- Architecture panel: exact prior 16-circuit panel. No architecture is added or removed based on Iris/Digits outcomes.

### Prespecified frozen decision gate

The protocol and thresholds below were frozen internally before this benchmark was evaluated; they were **not registered in a public preregistration registry**.

The final proxy benchmark is **PASS** only if all four conditions hold:

1. **H1:** E2Q Spearman \(\rho>0\) in at least 7/9 cells, \(\rho\ge0.40\) in at least 6/9, and the hierarchical pooled 95% CI lower bound is above 0.20.
2. **H2:** E2Q beats both \(N_{2q}\) and logical-depth correlation in at least 6/9 cells, the median cell-level advantage over the stronger structural baseline is at least 0.05, and a paired cell bootstrap gives a 90% CI lower bound above 0 for the mean Fisher-z advantage.
3. **H3:** within-architecture demeaned E2Q/distortion Spearman \(\rho>0\) in at least 6/9 cells, \(\rho\ge0.30\) in at least 5/9, and the hierarchical pooled 95% CI lower bound is above 0.10.
4. **Family robustness:** after leaving out one entanglement family at a time, E2Q remains positively associated with distortion in at least 75% of the 36 checks.

A **PARTIAL** verdict means H1 and H3 pass but at least one complexity-value-add/family criterion does not. Otherwise the verdict is **REVISE**. Thresholds are not changed after results are inspected.

In [ ]:
from pathlib import Path
import base64, hashlib, io, json, math, os, shutil, sys, time, zipfile, warnings

# ---------------- FROZEN CONFIG ----------------
DATASETS = ['breast_cancer','iris_binary','digits_01']
BACKENDS = ['ibm_fez','ibm_kingston','ibm_marrakesh']
PHYSICAL_PATH = [0,1,2,3]
CALIBRATION_IDXS = [0,2,4,7]
DATA_SPLIT_SEED = 17
TRAIN_SEEDS = [17,29,43,71,101]
NOISY_SEEDS = [17,43,101]
TRAIN_MAXITER = 20
MAX_FIT = 160
MAX_VAL = 80
MAX_TEST = 80
MAX_NOISY_TEST = 64
BOOTSTRAPS = 800
HIER_BOOTSTRAPS = 800
FAMILY_MIN_FRACTION = 0.75
REUSE_CHECKPOINTS = True

WORK_ROOT = Path('/content') if Path('/content').exists() else Path.cwd()
PROJECT_DIR = WORK_ROOT/'E2Q_Proxy_Full_Benchmark'
SOURCE_DIR = PROJECT_DIR/'source'
MODEL_DIR = PROJECT_DIR/'model_checkpoints'
PROB_DIR = PROJECT_DIR/'probability_checkpoints'
RESULT_DIR = PROJECT_DIR/'results'
FIG_DIR = RESULT_DIR/'figures'
for p in [PROJECT_DIR,SOURCE_DIR,MODEL_DIR,PROB_DIR,RESULT_DIR,FIG_DIR]: p.mkdir(parents=True,exist_ok=True)

print('Datasets:',DATASETS)
print('Backends:',BACKENDS)
print('Calibration indices:',CALIBRATION_IDXS)
print('Training seeds:',TRAIN_SEEDS)
print('Noisy validation seeds:',NOISY_SEEDS)
print('Embedded source SHA256:','11ab9a6ea044d9b37705e69ba6e8e5ba80a396507fca7a88fe3b897613a1727b')

### 1. Install the reproducible software stack

Qiskit/Aer are needed only for sparse four-qubit exact density-matrix validation. IBM Runtime is **not** required because the historical compact calibration snapshots are embedded from the previously audited runs.

In [ ]:
import subprocess
packages = [
    'qiskit>=2.5,<2.6',
    'qiskit-aer>=0.17,<0.18',
    'numpy>=2.0',
    'scipy>=1.12',
    'pandas>=2.0',
    'scikit-learn>=1.4',
    'matplotlib>=3.8',
    'tabulate>=0.9',
]
subprocess.check_call([sys.executable,'-m','pip','install','-q',*packages])
print('Dependencies installed.')

### 2. Restore and verify the frozen source bundle

The bundle contains the exact 64-library definition, the previously frozen 16-circuit panel, eight compact calibration snapshots for each backend, the prior cross-backend verdict that froze E2Q, and the original Breast Cancer seed-17 parameter bank for continuity checking. It contains no credential.

In [ ]:
SOURCE_ZIP_B64 = """..."""
raw = base64.b64decode(SOURCE_ZIP_B64)
actual_sha = hashlib.sha256(raw).hexdigest()
expected_sha = '11ab9a6ea044d9b37705e69ba6e8e5ba80a396507fca7a88fe3b897613a1727b'
assert actual_sha == expected_sha, (actual_sha, expected_sha)
with zipfile.ZipFile(io.BytesIO(raw)) as zf:
    zf.extractall(PROJECT_DIR)
# ZIP has one top-level e2q_full_source folder.
restored = PROJECT_DIR/'e2q_full_source'
if SOURCE_DIR.exists():
    for p in SOURCE_DIR.iterdir():
        if p.is_file(): p.unlink()
for p in restored.iterdir():
    shutil.copy2(p,SOURCE_DIR/p.name)
print('Source restored:',SOURCE_DIR)
print('SHA256:',actual_sha)
print(sorted(p.name for p in SOURCE_DIR.iterdir()))

## Data

### 3. Load the frozen architectures, panel, calibrations, and prior verdict

In [ ]:
import numpy as np
import pandas as pd
from dataclasses import dataclass

arch_json = json.loads((SOURCE_DIR/'architecture_library_64.json').read_text())
panel_source = pd.read_csv(SOURCE_DIR/'panel_16.csv')
panel_ids = panel_source['architecture_id'].tolist()
prior_verdict = json.loads((SOURCE_DIR/'crossbackend_verdict.json').read_text())
selector_verdict = json.loads((SOURCE_DIR/'selector_verdict.json').read_text())
assert prior_verdict['verdict']=='PASS' and prior_verdict['recommended_proxy']=='E2Q'
assert len(arch_json)==64 and len(panel_ids)==16 and len(set(panel_ids))==16

MANIFESTS = {b:json.loads((SOURCE_DIR/f'{b}_manifest_8.json').read_text()) for b in BACKENDS}
for b,m in MANIFESTS.items():
    assert m['physical_qubits']==PHYSICAL_PATH
    assert len(m['snapshots'])==8
    assert all(int(i)<len(m['snapshots']) for i in CALIBRATION_IDXS)

@dataclass(frozen=True)
class Arch:
    id:str
    rotations:tuple
    entanglement:str
    edges:tuple
    reps:int
    n_params:int
    twoq_count:int
    @property
    def sx_pulses_per_qubit(self):
        # One RY feature encoding + every RX/RY variational gate, each approximated as 2 SX pulses; RZ is virtual.
        return 2*(1 + self.reps*sum(r in ('rx','ry') for r in self.rotations))
    @property
    def edge_layer_depth(self):
        # Minimum parallel CZ layers for the frozen path-edge pattern.
        edges=list(self.edges)
        remaining=edges.copy(); layers=0
        while remaining:
            used=set(); nxt=[]
            for a,b in remaining:
                if a not in used and b not in used:
                    used|={a,b}
                else:
                    nxt.append((a,b))
            layers+=1; remaining=nxt
        return layers
    @property
    def logical_depth(self):
        return 1 + self.reps*(len(self.rotations)+self.edge_layer_depth)

ARCHS=[Arch(r['id'],tuple(r['rotations']),r['entanglement'],tuple(tuple(e) for e in r['edges']),int(r['reps']),int(r['n_params']),int(r['twoq_count'])) for r in arch_json]
ARCH_BY_ID={a.id:a for a in ARCHS}
PANEL=[ARCH_BY_ID[x] for x in panel_ids]
assert all(a.id in ARCH_BY_ID for a in PANEL)
print('Prior proxy verdict:',prior_verdict['verdict'],'recommended:',prior_verdict['recommended_proxy'])
print('Prior selector verdict:',selector_verdict.get('verdict'))
print('Frozen panel IDs:',panel_ids)
display(panel_source)

### 4. Prepare all three datasets with one fixed split

The split/preprocessing pipeline is identical across model seeds so that seed effects reflect parameter training only:

stratified split → train-only StandardScaler → train-only PCA(4) → train-only MinMax to \([-\pi,\pi]\) → deterministic stratified caps.

In [ ]:
from sklearn.datasets import load_breast_cancer, load_iris, load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.metrics import balanced_accuracy_score, f1_score, roc_auc_score, log_loss, brier_score_loss


def raw_dataset(name):
    if name=='breast_cancer':
        d=load_breast_cancer(); return d.data,d.target.astype(int)
    if name=='iris_binary':
        d=load_iris(); mask=d.target<2; return d.data[mask],d.target[mask].astype(int)
    if name=='digits_01':
        d=load_digits(); mask=d.target<2; return d.data[mask],d.target[mask].astype(int)
    raise ValueError(name)

def stratified_cap(X,y,cap,seed):
    if cap is None or cap>=len(y): return np.asarray(X),np.asarray(y)
    rr=np.random.default_rng(seed); idx=[]
    for c in np.unique(y):
        ids=np.flatnonzero(y==c); rr.shuffle(ids)
        take=max(1,int(round(cap*len(ids)/len(y)))); idx.extend(ids[:take].tolist())
    idx=np.array(idx,dtype=int)
    if len(idx)>cap:
        rr.shuffle(idx); idx=idx[:cap]
    elif len(idx)<cap:
        rem=np.setdiff1d(np.arange(len(y)),idx); rr.shuffle(rem); idx=np.r_[idx,rem[:cap-len(idx)]]
    idx=np.sort(idx); return np.asarray(X)[idx],np.asarray(y)[idx]

def prepare_dataset(name):
    X,y=raw_dataset(name)
    X_fitval,X_test,y_fitval,y_test=train_test_split(X,y,test_size=0.20,random_state=DATA_SPLIT_SEED,stratify=y)
    X_fit,X_val,y_fit,y_val=train_test_split(X_fitval,y_fitval,test_size=0.20,random_state=DATA_SPLIT_SEED,stratify=y_fitval)
    scaler=StandardScaler().fit(X_fit)
    xs_fit,xs_val,xs_test=scaler.transform(X_fit),scaler.transform(X_val),scaler.transform(X_test)
    pca=PCA(n_components=4,random_state=DATA_SPLIT_SEED).fit(xs_fit)
    xp_fit,xp_val,xp_test=pca.transform(xs_fit),pca.transform(xs_val),pca.transform(xs_test)
    angle=MinMaxScaler(feature_range=(-np.pi,np.pi),clip=True).fit(xp_fit)
    xa_fit,xa_val,xa_test=angle.transform(xp_fit),angle.transform(xp_val),angle.transform(xp_test)
    xa_fit,y_fit=stratified_cap(xa_fit,y_fit,MAX_FIT,DATA_SPLIT_SEED)
    xa_val,y_val=stratified_cap(xa_val,y_val,MAX_VAL,DATA_SPLIT_SEED+1)
    xa_test,y_test=stratified_cap(xa_test,y_test,MAX_TEST,DATA_SPLIT_SEED+2)
    x_noise,y_noise=stratified_cap(xa_test,y_test,MAX_NOISY_TEST,DATA_SPLIT_SEED+3)
    return {'name':name,'X_fit':xa_fit,'y_fit':y_fit,'X_val':xa_val,'y_val':y_val,'X_test':xa_test,'y_test':y_test,
            'X_noise':x_noise,'y_noise':y_noise,'pca_explained_variance':pca.explained_variance_ratio_.tolist()}

DATA={name:prepare_dataset(name) for name in DATASETS}
meta=[]
for name,d in DATA.items():
    meta.append({'dataset':name,'fit_n':len(d['y_fit']),'val_n':len(d['y_val']),'test_n':len(d['y_test']),
                 'noisy_test_n':len(d['y_noise']),'positive_rate_fit':float(np.mean(d['y_fit'])),
                 'pca_explained_variance_sum':float(np.sum(d['pca_explained_variance']))})
meta_df=pd.DataFrame(meta)
meta_df.to_csv(RESULT_DIR/'dataset_metadata.csv',index=False)
display(meta_df)

### 5. Define the exact ideal simulator and verify Breast Cancer continuity

In [ ]:
def rx(t):
    c,s=np.cos(t/2),-1j*np.sin(t/2); return np.array([[c,s],[s,c]],complex)
def ry(t):
    c,s=np.cos(t/2),np.sin(t/2); return np.array([[c,-s],[s,c]],complex)
def rz(t): return np.array([[np.exp(-.5j*t),0],[0,np.exp(.5j*t)]],complex)
ROT={'rx':rx,'ry':ry,'rz':rz}

def apply_single(st,mat,q):
    step=1<<q; span=step<<1
    for base in range(0,len(st),span):
        for off in range(step):
            i0=base+off; i1=i0+step; a,b=st[i0],st[i1]
            st[i0]=mat[0,0]*a+mat[0,1]*b; st[i1]=mat[1,0]*a+mat[1,1]*b

def apply_cz(st,a,b):
    ma,mb=1<<a,1<<b
    for i in range(len(st)):
        if (i&ma) and (i&mb): st[i]*=-1

def prob1_one(x,arch,theta):
    st=np.zeros(16,complex); st[0]=1
    for q in range(4): apply_single(st,ry(float(x[q])),q)
    k=0
    for _ in range(arch.reps):
        for r in arch.rotations:
            for q in range(4): apply_single(st,ROT[r](float(theta[k])),q); k+=1
        for a,b in arch.edges: apply_cz(st,a,b)
    probs=np.abs(st)**2
    return float(sum(p for i,p in enumerate(probs) if i&1))

def predict_probs(X,arch,theta):
    return np.asarray([prob1_one(x,arch,theta) for x in np.asarray(X)],float)

def bce(y,p):
    p=np.clip(np.asarray(p,float),1e-8,1-1e-8)
    return float(-np.mean(y*np.log(p)+(1-y)*np.log(1-p)))

def metric_bundle(y,p):
    p=np.clip(np.asarray(p,float),1e-9,1-1e-9); pred=(p>=.5).astype(int)
    out={'balanced_accuracy':float(balanced_accuracy_score(y,pred)),
         'f1':float(f1_score(y,pred,zero_division=0)),
         'logloss':float(log_loss(y,p,labels=[0,1])),
         'brier':float(brier_score_loss(y,p))}
    try: out['auc']=float(roc_auc_score(y,p))
    except ValueError: out['auc']=float('nan')
    return out

# Continuity check against the previously saved Breast Cancer seed-17 models.
old_params_npz=np.load(SOURCE_DIR/'breast_cancer_seed17_parameter_bank.npz')
OLD_PARAMS={k:np.asarray(old_params_npz[k],float) for k in old_params_npz.files}
old_summary=pd.read_csv(SOURCE_DIR/'breast_cancer_seed17_ideal_summary.csv')
verify=[]
for aid in panel_ids:
    p=predict_probs(DATA['breast_cancer']['X_test'],ARCH_BY_ID[aid],OLD_PARAMS[aid])
    b=metric_bundle(DATA['breast_cancer']['y_test'],p)['balanced_accuracy']
    src=float(old_summary.loc[old_summary.architecture_id==aid,'ideal_test_balanced_accuracy'].iloc[0])
    verify.append(abs(b-src))
assert max(verify)<1e-10, max(verify)
print('Breast Cancer seed-17 continuity PASS; max BACC discrepancy:',max(verify))

### 6. Train/checkpoint the frozen 16-circuit panel across five seeds

Seed 17 on Breast Cancer reuses the already-audited saved parameter bank. All other dataset/seed/panel combinations use the same 20-step random-coordinate training rule. Every model is saved independently, so interrupted Colab sessions can resume.

In [ ]:
def stable_model_seed(dataset_name,train_seed,aid):
    token=f'{dataset_name}|{train_seed}|{aid}'.encode()
    return int(hashlib.sha256(token).hexdigest()[:8],16)%(2**32-1)

def train_arch(arch,X,y,seed,maxiter=TRAIN_MAXITER):
    rr=np.random.default_rng(seed)
    theta=rr.uniform(-.25,.25,size=arch.n_params)
    best=bce(y,predict_probs(X,arch,theta)); step=.45
    for i in range(maxiter):
        prop=theta.copy(); j=int(rr.integers(0,len(theta))); prop[j]+=float(rr.normal(0,step))
        loss=bce(y,predict_probs(X,arch,prop))
        if loss<best: theta,best=prop,loss
        if (i+1)%5==0: step*=.8
    return theta,float(best)

def model_checkpoint(dataset,seed,aid):
    p=MODEL_DIR/dataset/str(seed); p.mkdir(parents=True,exist_ok=True)
    return p/f'{aid}.npz'

training_rows=[]; start=time.time(); total=len(DATASETS)*len(TRAIN_SEEDS)*len(PANEL); done=0
for dataset in DATASETS:
    d=DATA[dataset]
    for seed in TRAIN_SEEDS:
        for arch in PANEL:
            cp=model_checkpoint(dataset,seed,arch.id)
            if REUSE_CHECKPOINTS and cp.exists():
                z=np.load(cp); theta=np.asarray(z['theta'],float); train_loss=float(z['train_loss'])
            elif dataset=='breast_cancer' and seed==17:
                theta=OLD_PARAMS[arch.id].copy(); train_loss=bce(d['y_fit'],predict_probs(d['X_fit'],arch,theta))
                np.savez_compressed(cp,theta=theta,train_loss=train_loss,source='reused_prior_audited_seed17')
            else:
                theta,train_loss=train_arch(arch,d['X_fit'],d['y_fit'],stable_model_seed(dataset,seed,arch.id))
                np.savez_compressed(cp,theta=theta,train_loss=train_loss,source='trained_full_benchmark')
            fitm=metric_bundle(d['y_fit'],predict_probs(d['X_fit'],arch,theta))
            valm=metric_bundle(d['y_val'],predict_probs(d['X_val'],arch,theta))
            testm=metric_bundle(d['y_test'],predict_probs(d['X_test'],arch,theta))
            training_rows.append({'dataset':dataset,'seed':seed,'architecture_id':arch.id,'rotations':'+'.join(arch.rotations),
                                  'entanglement':arch.entanglement,'reps':arch.reps,'twoq_count':arch.twoq_count,
                                  'logical_depth':arch.logical_depth,'n_params':arch.n_params,'train_loss':train_loss,
                                  'fit_bacc':fitm['balanced_accuracy'],'val_bacc':valm['balanced_accuracy'],'test_bacc':testm['balanced_accuracy'],
                                  'test_auc':testm['auc'],'test_logloss':testm['logloss'],'test_brier':testm['brier']})
            done+=1
            if done%16==0 or done==total:
                print(f'Ideal training/checkpoint progress: {done}/{total} ({100*done/total:.1f}%)')

ideal_df=pd.DataFrame(training_rows)
ideal_df.to_csv(RESULT_DIR/'ideal_training_summary.csv',index=False)
print('Ideal stage complete in',round(time.time()-start,1),'s')
display(ideal_df.groupby('dataset')[['test_bacc','test_auc']].agg(['mean','std','min','max']))

### 7. Consolidate trained parameters for the result package

In [ ]:
all_params={}
for dataset in DATASETS:
    for seed in TRAIN_SEEDS:
        for aid in panel_ids:
            z=np.load(model_checkpoint(dataset,seed,aid))
            all_params[f'{dataset}__seed{seed}__{aid}']=np.asarray(z['theta'],float)
np.savez_compressed(RESULT_DIR/'trained_parameter_bank_panel16.npz',**all_params)
print('Consolidated parameter vectors:',len(all_params))

## Results

### 8. Compute E2Q and all prespecified proxy ablations

Comparators are static 2Q count, logical depth, the earlier OLD composite, and full CARE composite. E2Q itself remains unchanged.

In [ ]:
def neglog_survival(p):
    p=float(np.clip(p,0,1-1e-12)); return -math.log1p(-p)

def proxy_components(arch,snap):
    qrows=snap['qubits']; edge_lookup={tuple(e['local_edge']):e for e in snap['edges']}
    oneq=0.0; twoq=0.0; durations=np.zeros(4,float)
    pulses=arch.sx_pulses_per_qubit
    for q in range(4):
        oneq += pulses*neglog_survival(qrows[q]['sx_error'])
        durations[q] += pulses*float(qrows[q]['sx_length_s'])
    for _ in range(arch.reps):
        for edge in arch.edges:
            e=edge_lookup[tuple(edge)]
            twoq += neglog_survival(e['gate_error'])
            dur=float(e['gate_length_s'])
            for q in edge: durations[q]+=dur
    readout=neglog_survival(qrows[0]['readout_error'])
    old_coherence=relax=dephase=0.0
    for q in range(4):
        t1=float(qrows[q]['t1_s']); t2=float(qrows[q]['t2_s'])
        if not (t1>0 and t2>0): raise ValueError('Non-positive active T1/T2')
        old_coherence += durations[q]*(1/t1+1/t2)
        relax += durations[q]/t1
        inv_tphi=max(0.0,1/t2-1/(2*t1)); dephase += durations[q]*inv_tphi
    return {'N2Q':float(arch.twoq_count),'Depth':float(arch.logical_depth),'E2Q':float(twoq),
            'OLD':float(oneq+twoq+readout+old_coherence),
            'CARE':float(oneq+twoq+readout+relax+dephase),
            'oneq':float(oneq),'readout':float(readout),'relax':float(relax),'dephase':float(dephase)}

proxy_rows=[]; prov=[]
for b,m in MANIFESTS.items():
    for idx in CALIBRATION_IDXS:
        snap=m['snapshots'][idx]
        prov.append({'backend':b,'snapshot':idx,'requested_timestamp':snap['requested_timestamp'],
                     'returned_timestamp':snap['returned_timestamp'],'physical_qubits':str(m['physical_qubits']),
                     'raw_sha256':snap.get('raw_sha256','')})
        for arch in PANEL:
            proxy_rows.append({'backend':b,'snapshot':idx,'architecture_id':arch.id,'rotations':'+'.join(arch.rotations),
                               'entanglement':arch.entanglement,'reps':arch.reps,**proxy_components(arch,snap)})
proxy_df=pd.DataFrame(proxy_rows); prov_df=pd.DataFrame(prov)
proxy_df.to_csv(RESULT_DIR/'proxy_matrix.csv',index=False)
prov_df.to_csv(RESULT_DIR/'calibration_provenance.csv',index=False)
display(proxy_df.head(12)); display(prov_df)

### 9. Build only 12 sparse four-qubit historical noise models

The model matches the earlier validated pipeline:

- RX/RY error ≈ two SX exposures plus thermal relaxation;
- historical CZ depolarizing + thermal relaxation on the local path edge;
- RZ is virtual;
- the stored average readout error on logical qubit 0 is applied symmetrically after exact quantum probabilities are obtained.

This is an explicit approximation study, not a claim that Aer reproduces every device-level effect.

In [ ]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error, thermal_relaxation_error

def valid_t2(t1,t2): return min(float(t2),2.0*float(t1))
def sparse_noise_model(snap):
    nm=NoiseModel(basis_gates=['rx','ry','rz','cz'])
    for q,row in enumerate(snap['qubits']):
        t1=float(row['t1_s']); t2=valid_t2(t1,row['t2_s'])
        p_eff=1-(1-float(row['sx_error']))**2
        dur=2*float(row['sx_length_s'])
        dep=depolarizing_error(float(np.clip(p_eff,0,0.999999)),1)
        rel=thermal_relaxation_error(t1,t2,dur)
        nm.add_quantum_error(dep.compose(rel),['rx','ry'],[q])
    qrows=snap['qubits']
    for e in snap['edges']:
        i,j=map(int,e['local_edge']); dur=float(e['gate_length_s']); p=float(np.clip(e['gate_error'],0,0.999999))
        ti=thermal_relaxation_error(float(qrows[i]['t1_s']),valid_t2(qrows[i]['t1_s'],qrows[i]['t2_s']),dur)
        tj=thermal_relaxation_error(float(qrows[j]['t1_s']),valid_t2(qrows[j]['t1_s'],qrows[j]['t2_s']),dur)
        therm=tj.tensor(ti); dep=depolarizing_error(p,2)
        nm.add_quantum_error(dep.compose(therm),'cz',[i,j])
    return nm

def qiskit_prob_circuit(x,arch,theta):
    qc=QuantumCircuit(4)
    for q in range(4): qc.ry(float(x[q]),q)
    k=0
    for _ in range(arch.reps):
        for r in arch.rotations:
            for q in range(4): getattr(qc,r)(float(theta[k]),q); k+=1
        for a,b in arch.edges: qc.cz(a,b)
    qc.save_probabilities([0],label='probabilities')
    return qc

def apply_symmetric_readout(p1,error):
    e=float(np.clip(error,0,0.499999)); return (1-e)*p1+e*(1-p1)

NOISE_MODELS={b:{idx:sparse_noise_model(MANIFESTS[b]['snapshots'][idx]) for idx in CALIBRATION_IDXS} for b in BACKENDS}
print('Sparse models built:',sum(len(x) for x in NOISE_MODELS.values()))

### 10. Circuit-semantics preflight

Before the expensive loop, compare the NumPy ideal simulator with noiseless Qiskit probabilities on representative circuits/samples. Any mismatch is a hard stop.

In [ ]:
from qiskit_aer import AerSimulator
preflight=[]
for dataset in DATASETS:
    d=DATA[dataset]
    for aid in [panel_ids[0],panel_ids[len(panel_ids)//2],panel_ids[-1]]:
        arch=ARCH_BY_ID[aid]
        z=np.load(model_checkpoint(dataset,17,aid)); theta=np.asarray(z['theta'],float)
        x=d['X_noise'][0]
        np_p=prob1_one(x,arch,theta)
        qc=qiskit_prob_circuit(x,arch,theta)
        rr=AerSimulator(method='statevector').run([qc],shots=None).result()
        q_p=float(np.asarray(rr.data(0)['probabilities'],float)[1])
        preflight.append(abs(np_p-q_p))
assert max(preflight)<1e-9, max(preflight)
print('Circuit-semantics preflight PASS; max probability discrepancy:',max(preflight))

### 11. Exact noisy benchmark — checkpointed and batched

Each Aer job contains the whole 16-architecture panel for one dataset × trained seed × backend × calibration state. This reduces simulator overhead to **108 bounded jobs** rather than thousands of separate runs. No shot sampling is used.

In [ ]:
def batch_checkpoint(dataset,seed,backend,snapshot):
    p=PROB_DIR/dataset/str(seed)/backend; p.mkdir(parents=True,exist_ok=True)
    return p/f'snapshot_{snapshot:02d}.npz'

def load_theta(dataset,seed,aid):
    return np.asarray(np.load(model_checkpoint(dataset,seed,aid))['theta'],float)

def run_noisy_batch(dataset,seed,backend,snapshot):
    cp=batch_checkpoint(dataset,seed,backend,snapshot)
    d=DATA[dataset]; X=d['X_noise']; y=d['y_noise']
    if REUSE_CHECKPOINTS and cp.exists():
        z=np.load(cp,allow_pickle=False)
        assert list(z['panel_ids'].astype(str))==panel_ids
        return z['ideal'],z['noisy'],z['y']
    circuits=[]; ideal=np.zeros((len(PANEL),len(X)),float)
    for ai,arch in enumerate(PANEL):
        theta=load_theta(dataset,seed,arch.id)
        ideal[ai]=predict_probs(X,arch,theta)
        circuits.extend(qiskit_prob_circuit(x,arch,theta) for x in X)
    sim=AerSimulator(noise_model=NOISE_MODELS[backend][snapshot],method='density_matrix')
    result=sim.run(circuits,shots=None).result()
    noisy_quant=np.zeros_like(ideal); k=0
    for ai in range(len(PANEL)):
        for si in range(len(X)):
            arr=np.asarray(result.data(k)['probabilities'],float); noisy_quant[ai,si]=float(arr[1]); k+=1
    ro=float(MANIFESTS[backend]['snapshots'][snapshot]['qubits'][0]['readout_error'])
    noisy=(1-ro)*noisy_quant + ro*(1-noisy_quant)
    np.savez_compressed(cp,panel_ids=np.asarray(panel_ids),ideal=ideal,noisy=noisy,y=np.asarray(y,int))
    return ideal,noisy,np.asarray(y,int)

# Single noise-model smoke test before the full loop.
_ip,_npb,_yy=run_noisy_batch(DATASETS[0],NOISY_SEEDS[0],BACKENDS[0],CALIBRATION_IDXS[0])
print('Noisy batch preflight PASS. Mean panel distortion:',float(np.mean(np.abs(_ip-_npb))))

In [ ]:
validation_rows=[]; sample_rows=[]
total=len(DATASETS)*len(NOISY_SEEDS)*len(BACKENDS)*len(CALIBRATION_IDXS); n_done=0; start=time.time()
for dataset in DATASETS:
    d=DATA[dataset]
    for seed in NOISY_SEEDS:
        for backend in BACKENDS:
            for snapshot in CALIBRATION_IDXS:
                ideal,noisy,y=run_noisy_batch(dataset,seed,backend,snapshot)
                for ai,arch in enumerate(PANEL):
                    im=metric_bundle(y,ideal[ai]); nm=metric_bundle(y,noisy[ai])
                    validation_rows.append({'dataset':dataset,'seed':seed,'backend':backend,'snapshot':snapshot,
                        'architecture_id':arch.id,'entanglement':arch.entanglement,'reps':arch.reps,'twoq_count':arch.twoq_count,
                        'logical_depth':arch.logical_depth,'probability_mae':float(np.mean(np.abs(ideal[ai]-noisy[ai]))),
                        'probability_rmse':float(np.sqrt(np.mean((ideal[ai]-noisy[ai])**2))),
                        'ideal_bacc':im['balanced_accuracy'],'noisy_bacc':nm['balanced_accuracy'],'delta_bacc':im['balanced_accuracy']-nm['balanced_accuracy'],
                        'ideal_auc':im['auc'],'noisy_auc':nm['auc'],'delta_auc':im['auc']-nm['auc'],
                        'delta_logloss':nm['logloss']-im['logloss'],'delta_brier':nm['brier']-im['brier']})
                    for si in range(len(y)):
                        sample_rows.append({'dataset':dataset,'seed':seed,'backend':backend,'snapshot':snapshot,'architecture_id':arch.id,
                                            'sample_idx':si,'y':int(y[si]),'ideal_prob':float(ideal[ai,si]),'noisy_prob':float(noisy[ai,si]),
                                            'abs_distortion':float(abs(ideal[ai,si]-noisy[ai,si]))})
                n_done+=1
                if n_done%6==0 or n_done==total:
                    print(f'Noisy benchmark progress: {n_done}/{total} ({100*n_done/total:.1f}%) | elapsed {(time.time()-start)/60:.1f} min')

validation_df=pd.DataFrame(validation_rows)
sample_df=pd.DataFrame(sample_rows)
validation_df.to_csv(RESULT_DIR/'noisy_validation_architecture_seed.csv',index=False)
sample_df.to_csv(RESULT_DIR/'noisy_probability_records.csv.gz',index=False,compression='gzip')
print('Architecture-level noisy rows:',len(validation_df),'sample rows:',len(sample_df))

### 12. Create the seed-averaged confirmatory analysis table

Training-seed replicates are averaged **before** proxy correlations so that repeated trained parameterizations do not artificially inflate the architecture/calibration sample size.

In [ ]:
analysis_df=(validation_df.groupby(['dataset','backend','snapshot','architecture_id','entanglement','reps','twoq_count','logical_depth'],as_index=False)
             .agg(probability_mae=('probability_mae','mean'),probability_mae_seed_sd=('probability_mae','std'),
                  probability_rmse=('probability_rmse','mean'),delta_logloss=('delta_logloss','mean'),delta_brier=('delta_brier','mean'),
                  delta_auc=('delta_auc','mean'),delta_bacc=('delta_bacc','mean')))
analysis_df=analysis_df.merge(proxy_df,on=['backend','snapshot','architecture_id','entanglement','reps'],how='left',validate='many_to_one')
assert len(analysis_df)==len(DATASETS)*len(BACKENDS)*len(CALIBRATION_IDXS)*len(PANEL)
analysis_df.to_csv(RESULT_DIR/'analysis_table_seed_averaged.csv',index=False)
display(analysis_df.head())

### 13. Per-cell proxy validity with architecture-block bootstrap

In [ ]:
from scipy.stats import rankdata, kendalltau

PROXIES=['N2Q','Depth','E2Q','OLD','CARE']

def safe_spearman(x,y):
    x=np.asarray(x,float); y=np.asarray(y,float)
    if len(x)<3 or np.ptp(x)==0 or np.ptp(y)==0: return np.nan
    rx=rankdata(x,method='average'); ry=rankdata(y,method='average')
    r=np.corrcoef(rx,ry)[0,1]
    return float(r) if np.isfinite(r) else np.nan

def safe_kendall(x,y):
    r=kendalltau(np.asarray(x,float),np.asarray(y,float)).statistic
    return float(r) if np.isfinite(r) else np.nan

def arch_block_bootstrap(df,xcol,ycol='probability_mae',B=BOOTSTRAPS,seed=17):
    archs=sorted(df.architecture_id.unique())
    # Fixed four-snapshot blocks; array form avoids expensive pandas reconstruction inside bootstrap.
    x_blocks=[]; y_blocks=[]
    for a in archs:
        g=df[df.architecture_id==a].sort_values('snapshot')
        x_blocks.append(g[xcol].to_numpy(float)); y_blocks.append(g[ycol].to_numpy(float))
    x_blocks=np.asarray(x_blocks,float); y_blocks=np.asarray(y_blocks,float)
    rr=np.random.default_rng(seed); vals=[]; A=len(archs)
    for _ in range(B):
        ids=rr.integers(0,A,size=A)
        r=safe_spearman(x_blocks[ids].reshape(-1),y_blocks[ids].reshape(-1))
        if np.isfinite(r): vals.append(r)
    if not vals: return (np.nan,np.nan)
    return tuple(np.quantile(vals,[0.025,0.975]))

corr_rows=[]
for dataset in DATASETS:
    for backend in BACKENDS:
        cell=analysis_df[(analysis_df.dataset==dataset)&(analysis_df.backend==backend)].copy()
        for p in PROXIES:
            if p=='E2Q':
                lo,hi=arch_block_bootstrap(cell,p,seed=1000+DATASETS.index(dataset)*100+BACKENDS.index(backend)*10+PROXIES.index(p))
            else:
                lo,hi=np.nan,np.nan
            corr_rows.append({'dataset':dataset,'backend':backend,'proxy':p,
                              'spearman_rho':safe_spearman(cell[p],cell.probability_mae),
                              'kendall_tau':safe_kendall(cell[p],cell.probability_mae),
                              'ci95_low':lo,'ci95_high':hi,'n_architectures':cell.architecture_id.nunique(),'n_arch_snapshot':len(cell)})
corr_df=pd.DataFrame(corr_rows)
corr_df.to_csv(RESULT_DIR/'cell_proxy_correlations.csv',index=False)
display(corr_df[corr_df.proxy=='E2Q'])

### 14. H3: within-architecture temporal sensitivity

For each dataset/backend cell, subtract each architecture's own mean E2Q and own mean distortion across the four calibration states, then correlate the residual changes. This removes static between-architecture complexity as the explanation for H3.

In [ ]:
def demean_by_arch(df,col):
    return df[col]-df.groupby('architecture_id')[col].transform('mean')

temporal_rows=[]
for dataset in DATASETS:
    for backend in BACKENDS:
        cell=analysis_df[(analysis_df.dataset==dataset)&(analysis_df.backend==backend)].copy()
        for p in PROXIES:
            cell['_x']=demean_by_arch(cell,p); cell['_y']=demean_by_arch(cell,'probability_mae')
            if p=='E2Q':
                lo,hi=arch_block_bootstrap(cell.rename(columns={'_x':'X','_y':'Y'}),'X','Y',B=BOOTSTRAPS,
                                           seed=3000+DATASETS.index(dataset)*100+BACKENDS.index(backend)*10+PROXIES.index(p))
            else:
                lo,hi=np.nan,np.nan
            temporal_rows.append({'dataset':dataset,'backend':backend,'proxy':p,
                                  'demeaned_spearman_rho':safe_spearman(cell['_x'],cell['_y']),
                                  'demeaned_kendall_tau':safe_kendall(cell['_x'],cell['_y']),
                                  'ci95_low':lo,'ci95_high':hi})
temporal_df=pd.DataFrame(temporal_rows)
temporal_df.to_csv(RESULT_DIR/'temporal_demeaned_correlations.csv',index=False)
display(temporal_df[temporal_df.proxy=='E2Q'])

### 15. Leave-one-entanglement-family-out robustness

In [ ]:
family_rows=[]
families=sorted(analysis_df.entanglement.unique())
for dataset in DATASETS:
    for backend in BACKENDS:
        cell=analysis_df[(analysis_df.dataset==dataset)&(analysis_df.backend==backend)]
        for fam in families:
            z=cell[cell.entanglement!=fam]
            family_rows.append({'dataset':dataset,'backend':backend,'left_out_family':fam,
                                'spearman_rho':safe_spearman(z.E2Q,z.probability_mae),
                                'n_architectures':z.architecture_id.nunique(),'n_rows':len(z)})
family_df=pd.DataFrame(family_rows)
family_df.to_csv(RESULT_DIR/'leave_one_entanglement_family_out.csv',index=False)
display(family_df.head(12))

### 16. Hierarchical pooled summaries and H2 paired complexity-value-add test

In [ ]:
def fisher_z(r):
    if not np.isfinite(r): return np.nan
    return float(np.arctanh(np.clip(r,-0.999999,0.999999)))
def inv_fisher(z): return float(np.tanh(z))

def nested_pooled_bootstrap(proxy,temporal=False,B=HIER_BOOTSTRAPS,seed=8080):
    cells=[(d,b) for d in DATASETS for b in BACKENDS]
    # Precompute architecture x snapshot arrays once for every cell.
    cache={}
    for d,b in cells:
        z=analysis_df[(analysis_df.dataset==d)&(analysis_df.backend==b)].copy()
        archs=sorted(z.architecture_id.unique())
        xb=[]; yb=[]
        for a in archs:
            g=z[z.architecture_id==a].sort_values('snapshot')
            x=g[proxy].to_numpy(float); y=g.probability_mae.to_numpy(float)
            if temporal:
                x=x-x.mean(); y=y-y.mean()
            xb.append(x); yb.append(y)
        cache[(d,b)]=(np.asarray(xb,float),np.asarray(yb,float))
    rr=np.random.default_rng(seed); reps=[]
    for _ in range(B):
        picked=[cells[i] for i in rr.integers(0,len(cells),size=len(cells))]
        zs=[]
        for key in picked:
            xb,yb=cache[key]; A=len(xb); ids=rr.integers(0,A,size=A)
            r=safe_spearman(xb[ids].reshape(-1),yb[ids].reshape(-1))
            if np.isfinite(r): zs.append(fisher_z(r))
        if zs: reps.append(inv_fisher(float(np.mean(zs))))
    if not reps: return (np.nan,np.nan,np.nan)
    return tuple(np.quantile(reps,[0.025,0.5,0.975]))

pooled_rows=[]
for p in PROXIES:
    point=corr_df[corr_df.proxy==p].spearman_rho.dropna().to_numpy(float)
    tpoint=temporal_df[temporal_df.proxy==p].demeaned_spearman_rho.dropna().to_numpy(float)
    pmed=inv_fisher(float(np.mean([fisher_z(r) for r in point]))) if len(point) else np.nan
    tpmed=inv_fisher(float(np.mean([fisher_z(r) for r in tpoint]))) if len(tpoint) else np.nan
    if p=='E2Q':
        lo,med,hi=nested_pooled_bootstrap(p,False,seed=8000+PROXIES.index(p))
        tlo,tmed,thi=nested_pooled_bootstrap(p,True,seed=9000+PROXIES.index(p))
    else:
        lo,med,hi=np.nan,pmed,np.nan
        tlo,tmed,thi=np.nan,tpmed,np.nan
    pooled_rows.append({'proxy':p,'pooled_rho_median':med,'pooled_ci95_low':lo,'pooled_ci95_high':hi,
                        'temporal_rho_median':tmed,'temporal_ci95_low':tlo,'temporal_ci95_high':thi})
pooled_df=pd.DataFrame(pooled_rows)
pooled_df.to_csv(RESULT_DIR/'hierarchical_pooled_proxy_summary.csv',index=False)
display(pooled_df)

# H2 cell-level advantage over the stronger static structural baseline.
piv=corr_df.pivot_table(index=['dataset','backend'],columns='proxy',values='spearman_rho')
piv['structural_best']=piv[['N2Q','Depth']].max(axis=1)
piv['E2Q_advantage']=piv['E2Q']-piv['structural_best']
adv=piv['E2Q_advantage'].to_numpy(float)
rr=np.random.default_rng(12345); boot=[]
for _ in range(10000):
    x=rr.choice(adv,size=len(adv),replace=True)
    boot.append(float(np.mean([fisher_z(np.clip(piv.E2Q.iloc[i],-.999999,.999999))-fisher_z(np.clip(piv.structural_best.iloc[i],-.999999,.999999)) for i in rr.integers(0,len(piv),size=len(piv))])))
adv_ci90=np.quantile(boot,[0.05,0.95])
piv.reset_index().to_csv(RESULT_DIR/'h2_complexity_value_add.csv',index=False)
print('H2 cell comparison:')
display(piv.reset_index())
print('Median raw rho advantage:',float(np.median(adv)))
print('90% bootstrap CI for mean Fisher-z advantage:',adv_ci90)

### 17. Evaluate H1–H3 and issue the frozen verdict

In [ ]:
e2q_cells=corr_df[corr_df.proxy=='E2Q'].copy()
e2q_temp=temporal_df[temporal_df.proxy=='E2Q'].copy()
e2q_pool=pooled_df[pooled_df.proxy=='E2Q'].iloc[0]

h1_pos=int((e2q_cells.spearman_rho>0).sum())
h1_strong=int((e2q_cells.spearman_rho>=0.40).sum())
H1=bool(h1_pos>=7 and h1_strong>=6 and float(e2q_pool.pooled_ci95_low)>0.20)

h2_better=int((piv.E2Q>piv.structural_best).sum())
h2_median=float(np.median(piv.E2Q_advantage))
H2=bool(h2_better>=6 and h2_median>=0.05 and float(adv_ci90[0])>0)

h3_pos=int((e2q_temp.demeaned_spearman_rho>0).sum())
h3_strong=int((e2q_temp.demeaned_spearman_rho>=0.30).sum())
H3=bool(h3_pos>=6 and h3_strong>=5 and float(e2q_pool.temporal_ci95_low)>0.10)

family_positive=int((family_df.spearman_rho>0).sum()); family_total=len(family_df)
FAMILY=bool(family_positive/family_total>=FAMILY_MIN_FRACTION)

if H1 and H2 and H3 and FAMILY:
    verdict='PASS'
elif H1 and H3:
    verdict='PARTIAL'
else:
    verdict='REVISE'

verdict_obj={
    'verdict':verdict,'proxy':'E2Q','H1_predictive_validity':H1,'H2_calibration_value_add':H2,'H3_temporal_sensitivity':H3,
    'family_robustness':FAMILY,'h1_positive_cells':h1_pos,'h1_rho_ge_0_40_cells':h1_strong,
    'h2_cells_beating_both_structural_baselines':h2_better,'h2_median_rho_advantage':h2_median,
    'h2_fisher_z_advantage_ci90':list(map(float,adv_ci90)),'h3_positive_cells':h3_pos,'h3_rho_ge_0_30_cells':h3_strong,
    'family_positive_checks':family_positive,'family_total_checks':family_total,
    'pooled_e2q_rho':float(e2q_pool.pooled_rho_median),'pooled_e2q_ci95':[float(e2q_pool.pooled_ci95_low),float(e2q_pool.pooled_ci95_high)],
    'pooled_temporal_rho':float(e2q_pool.temporal_rho_median),'pooled_temporal_ci95':[float(e2q_pool.temporal_ci95_low),float(e2q_pool.temporal_ci95_high)],
    'frozen_gate':'H1 & H2 & H3 & family robustness',
}
(RESULT_DIR/'benchmark_verdict.json').write_text(json.dumps(verdict_obj,indent=2))
print(json.dumps(verdict_obj,indent=2))

### 18. Publication figures

In [ ]:
import matplotlib.pyplot as plt

# Figure 1: E2Q rho heatmap.
heat=e2q_cells.pivot(index='dataset',columns='backend',values='spearman_rho').reindex(index=DATASETS,columns=BACKENDS)
fig,ax=plt.subplots(figsize=(7.6,4.6)); im=ax.imshow(heat.values,aspect='auto',vmin=-1,vmax=1,cmap='coolwarm')
ax.set_xticks(range(len(BACKENDS)),BACKENDS,rotation=15); ax.set_yticks(range(len(DATASETS)),DATASETS)
for i in range(len(DATASETS)):
    for j in range(len(BACKENDS)): ax.text(j,i,f'{heat.iloc[i,j]:.2f}',ha='center',va='center')
ax.set_title('E2Q vs noisy probability distortion: Spearman correlation'); fig.colorbar(im,ax=ax,label='Spearman rho')
fig.tight_layout(); fig.savefig(FIG_DIR/'fig1_e2q_cell_correlations.png',dpi=220); plt.show()

# Figure 2: mean proxy rho across the 9 cells.
proxy_mean=corr_df.groupby('proxy').spearman_rho.agg(['mean','std']).reindex(PROXIES)
fig,ax=plt.subplots(figsize=(7.2,4.4)); ax.bar(proxy_mean.index,proxy_mean['mean'],yerr=proxy_mean['std'],capsize=4)
ax.axhline(0,lw=0.8); ax.set_ylabel('Mean cell Spearman rho'); ax.set_title('Proxy comparison across 9 dataset-backend cells')
fig.tight_layout(); fig.savefig(FIG_DIR/'fig2_proxy_comparison.png',dpi=220); plt.show()

# Figure 3: H3 temporal sensitivity heatmap.
temp=e2q_temp.pivot(index='dataset',columns='backend',values='demeaned_spearman_rho').reindex(index=DATASETS,columns=BACKENDS)
fig,ax=plt.subplots(figsize=(7.6,4.6)); im=ax.imshow(temp.values,aspect='auto',vmin=-1,vmax=1,cmap='coolwarm')
ax.set_xticks(range(len(BACKENDS)),BACKENDS,rotation=15); ax.set_yticks(range(len(DATASETS)),DATASETS)
for i in range(len(DATASETS)):
    for j in range(len(BACKENDS)): ax.text(j,i,f'{temp.iloc[i,j]:.2f}',ha='center',va='center')
ax.set_title('Within-architecture temporal E2Q sensitivity'); fig.colorbar(im,ax=ax,label='Demeaned Spearman rho')
fig.tight_layout(); fig.savefig(FIG_DIR/'fig3_temporal_sensitivity.png',dpi=220); plt.show()

# Figure 4: forest plot for E2Q per-cell estimates.
z=e2q_cells.copy(); z['label']=z.dataset+' | '+z.backend; z=z.sort_values(['dataset','backend']).reset_index(drop=True)
y=np.arange(len(z)); fig,ax=plt.subplots(figsize=(8.4,6.0))
ax.errorbar(z.spearman_rho,y,xerr=[z.spearman_rho-z.ci95_low,z.ci95_high-z.spearman_rho],fmt='o',capsize=3)
ax.axvline(0,lw=.8); ax.set_yticks(y,z.label); ax.set_xlabel('Spearman rho (95% architecture-block bootstrap CI)')
ax.set_title('E2Q predictive validity by dataset and backend'); fig.tight_layout(); fig.savefig(FIG_DIR/'fig4_e2q_forest.png',dpi=220); plt.show()

### 19. Generate the audit report and result ZIP

In [ ]:
# Compact report tables.
e2q_table=e2q_cells[['dataset','backend','spearman_rho','kendall_tau','ci95_low','ci95_high']].copy()
temp_table=e2q_temp[['dataset','backend','demeaned_spearman_rho','ci95_low','ci95_high']].copy()
proxy_table=corr_df.groupby('proxy',as_index=False).agg(mean_rho=('spearman_rho','mean'),median_rho=('spearman_rho','median'),min_rho=('spearman_rho','min'),max_rho=('spearman_rho','max'))

lines=[]
lines += ['# E2Q Proxy Full Benchmark Report','']
lines += [f"**Verdict:** `{verdict}`",'']
lines += ['## Frozen hypotheses','',f'- H1 predictive validity: **{H1}**',f'- H2 calibration value-add beyond N2Q/depth: **{H2}**',f'- H3 within-architecture temporal sensitivity: **{H3}**',f'- Leave-one-family robustness: **{FAMILY}**','']
lines += ['## E2Q predictive validity by dataset/backend','',e2q_table.to_markdown(index=False,floatfmt='.4f'),'']
lines += ['## E2Q within-architecture temporal sensitivity','',temp_table.to_markdown(index=False,floatfmt='.4f'),'']
lines += ['## Proxy comparison across nine cells','',proxy_table.to_markdown(index=False,floatfmt='.4f'),'']
lines += ['## Pooled summary','',pooled_df.to_markdown(index=False,floatfmt='.4f'),'']
lines += ['## H2 complexity comparison','',piv.reset_index().to_markdown(index=False,floatfmt='.4f'),'']
lines += ['## Interpretation rules','']
if verdict=='PASS':
    lines += ['The frozen E2Q proxy satisfies all prespecified cross-dataset validity, calibration-value-add, temporal-sensitivity, and family-robustness criteria. Proceed to manuscript preparation; direct real-QPU confirmation is optional future work and is not part of this release.']
elif verdict=='PARTIAL':
    lines += ['E2Q retains predictive validity and temporal sensitivity, but at least one prespecified value-add/family criterion is not met. Manuscript claims must be narrowed; do not claim superiority over structural complexity without resolving the failed criterion.']
else:
    lines += ['The frozen confirmatory gate is not met. Do not promote E2Q as a broadly validated calibration-aware proxy without revising the claim or experimental scope.']
lines += ['','## Scope caveats','',
          '- Noise validation uses sparse four-qubit Aer models reconstructed from historical compact IBM calibration data, not real-QPU executions.',
          '- Readout is approximated symmetrically from the stored average readout error.',
          '- The architecture panel is the exact 16-circuit panel frozen in the prior proxy-validation stage.',
          '- Exact density-matrix probabilities remove shot noise; training-seed variability is handled separately.']
report='\n'.join(lines)
(RESULT_DIR/'E2Q_FULL_BENCHMARK_REPORT.md').write_text(report,encoding='utf-8')
print(report[:7000])

config={'datasets':DATASETS,'backends':BACKENDS,'physical_path':PHYSICAL_PATH,'calibration_indices':CALIBRATION_IDXS,
        'data_split_seed':DATA_SPLIT_SEED,'train_seeds':TRAIN_SEEDS,'noisy_seeds':NOISY_SEEDS,'train_maxiter':TRAIN_MAXITER,
        'max_fit':MAX_FIT,'max_val':MAX_VAL,'max_test':MAX_TEST,'max_noisy_test':MAX_NOISY_TEST,
        'panel_ids':panel_ids,'source_bundle_sha256':expected_sha,'proxy':'E2Q','decision_gate':verdict_obj['frozen_gate']}
(RESULT_DIR/'benchmark_config.json').write_text(json.dumps(config,indent=2))

# Add frozen source manifests/panel for auditability without credentials.
for name in ['panel_16.csv','architecture_library_64.json','ibm_fez_manifest_8.json','ibm_kingston_manifest_8.json','ibm_marrakesh_manifest_8.json','crossbackend_verdict.json','SOURCE_PROVENANCE.json']:
    shutil.copy2(SOURCE_DIR/name,RESULT_DIR/f'source_{name}')

zip_path=WORK_ROOT/'E2Q_Proxy_Full_Benchmark.zip'
if zip_path.exists(): zip_path.unlink()
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as zf:
    for p in RESULT_DIR.rglob('*'):
        if p.is_file(): zf.write(p,p.relative_to(RESULT_DIR))
print('Result ZIP:',zip_path,'size MB',round(zip_path.stat().st_size/1024/1024,2))
try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    pass

## Takeaways

After execution, use `results/E2Q_FULL_BENCHMARK_REPORT.md` and `benchmark_verdict.json` as the authoritative readout. Do not reinterpret a failed prespecified criterion by changing thresholds or excluding a dataset/backend post hoc.

- **PASS:** proceed to a small real-QPU confirmation and manuscript preparation.
- **PARTIAL:** preserve the supported predictive/temporal claim but narrow any superiority claim; audit the failed H2/family condition before writing the paper.
- **REVISE:** stop scaling and revise the claim rather than adding more complexity to the proxy.